# OpenCV 实时颜色识别与目标跟踪

## 项目简介

本项目实现一个由 PC 或树莓派负责视觉识别、Arduino 负责舵机执行的实时颜色识别与目标跟踪系统。上位机通过 OpenCV 进行摄像头采集、HSV 阈值分割和目标中心定位，再通过串口把目标坐标发送给 Arduino，驱动双轴云台持续跟踪目标。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| PC 或树莓派 | 1 | 运行 OpenCV 视觉算法 |
| USB 摄像头 | 1 | 采集实时画面 |
| Arduino Uno | 1 | 接收目标坐标并控制云台 |
| 舵机 | 2 | 水平与俯仰跟踪 |
| 云台结构 | 1 套 | 安装摄像头或激光模块 |


## 核心功能

- OpenCV 进行颜色阈值分割。
- 计算目标轮廓中心坐标。
- 通过串口把目标中心发送给 Arduino。
- Arduino 根据偏差驱动双舵机跟踪。
- 丢失目标时自动进入扫描搜索模式。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 水平舵机 | D9 |
| 俯仰舵机 | D10 |
| 状态 LED | D13 |
| 上位机串口通信 | USB Serial |


## 接线说明

- 两路舵机信号线分别接 D9 和 D10，电源建议独立供电并与 Arduino 共地。
- 上位机通过 USB 串口与 Arduino 通信，波特率与代码中的 `115200` 保持一致。
- 状态 LED 接 D13，可用于直观看到当前是否处于跟踪状态。
- 摄像头连接在 PC 或树莓派侧，由 OpenCV 读取，不直接接到 Arduino。


## 串口协议

- 上位机发送格式：`T,found,cx,cy`。
- 示例：`T,1,145,118` 表示已找到目标，中心坐标为 `(145,118)`。
- 当 `found=0` 时，可继续发送 `T,0,0,0`，Arduino 会在超时后自动进入搜索模式。
- Arduino 会回传 `TRACK,cx,cy,pan,tilt`，便于上位机记录实际云台姿态。


## 运行方式

1. 打开 `src/opencv_real_time_color_recognition_and_target_tracking/opencv_real_time_color_recognition_and_target_tracking.ino` 并上传到 Arduino。
2. 上位机端运行 OpenCV 程序，完成 HSV 阈值分割和目标轮廓中心计算。
3. 将视觉程序输出的目标中心按协议发送到串口。
4. 调整 `FRAME_CENTER_X`、`FRAME_CENTER_Y` 使其与摄像头分辨率匹配。
5. 调整死区和比例步长，减少云台在中心附近的抖动。


## 控制逻辑说明

- `readSerialFrame()` 持续读取串口帧，直到遇到换行符再进入解析。
- `parseTrackingFrame()` 只接受 `T,found,cx,cy` 协议，保证视觉端和执行端格式统一。
- `updateTracking()` 根据目标中心与画面中心的偏差调整水平和俯仰角。
- 当偏差落在死区内时不再微调，避免舵机来回小幅抖动。
- 若超过超时时间没有收到有效目标，`runSearchPattern()` 会驱动云台左右扫描重新找目标。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 颜色识别 | 上位机能稳定输出目标中心 |
| 坐标发送 | Arduino 串口能持续接收目标帧 |
| 云台跟踪 | 目标移动时云台能持续追随 |
| 丢失搜索 | 目标消失后云台进入自动扫描 |


## 可扩展方向

- 增加 PID 云台控制，提升跟踪平滑度。
- 增加多颜色目标选择与优先级逻辑。
- 在上位机端叠加目标框、轨迹线和 FPS 显示。
- 将串口链路替换为 WiFi 或蓝牙无线控制。
